In [1]:

import logging
from pathlib import Path

logger = logging.getLogger("day14_pipeline")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter("%(levelname)s | %(message)s")
    )
    logger.addHandler(handler)

data_folder = Path(r"C:\tap-python-git\data")

required_files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

logger.info("Day 14 data validation started")

for filename in required_files:
    file_path = data_folder / filename

    if not file_path.exists():
        logger.error("File not found: %s", filename)
    elif file_path.stat().st_size == 0:
        logger.warning("Empty file: %s", filename)
    else:
        logger.info("File ready: %s", filename)

logger.info("File checking completed")


INFO | Day 14 data validation started
INFO | File ready: olist_customers_dataset.csv
INFO | File ready: olist_geolocation_dataset.csv
INFO | File ready: olist_orders_dataset.csv
INFO | File ready: olist_order_items_dataset.csv
INFO | File ready: olist_order_payments_dataset.csv
INFO | File ready: olist_order_reviews_dataset.csv
INFO | File ready: olist_products_dataset.csv
INFO | File ready: olist_sellers_dataset.csv
INFO | File ready: product_category_name_translation.csv
INFO | File checking completed


In [7]:

import logging
from pathlib import Path
from datetime import datetime

log_folder = Path("logs")
log_folder.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
log_file = log_folder / f"day14_{timestamp}.log"

logger = logging.getLogger("day14")
logger.setLevel(logging.INFO)
logger.propagate = False

for handler in logger.handlers[:]:
    if isinstance(handler, logging.FileHandler):
        logger.removeHandler(handler)
        handler.close()

file_handler = logging.FileHandler(log_file, encoding="utf-8")

file_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )
)

logger.addHandler(file_handler)

logger.info("Day 14 pipeline started")

for filename in files:
    path = data_folder / filename

    if not path.exists():
        logger.error("Missing file: %s", filename)
    elif path.stat().st_size == 0:
        logger.warning("Empty file: %s", filename)
    else:
        logger.info("File ready: %s", filename)

logger.info("Pipeline file check completed")

file_handler.flush()

print("Log file created:", log_file.resolve())


Log file created: C:\tap-python-git\logs\day14_20261123_142251_314955.log


In [6]:

from pathlib import Path

data_folder = Path(r"C:\tap-python-git\data")

files = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

print("Total files:", len(files))
print("Data folder:", data_folder)


Total files: 9
Data folder: C:\tap-python-git\data


In [1]:

%pip install pyyaml


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import yaml

print("PyYAML version:", yaml.__version__)

PyYAML version: 6.0.3


In [4]:
import yaml
from pathlib import Path

config_path = Path(r"C:\tap-python-git\config.yaml")

with open(config_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

print(config)

{'paths': {'data_dir': 'data', 'log_dir': 'logs', 'output_dir': 'output'}, 'thresholds': {'max_rejection_rate': 0.2, 'min_price': 0}, 'column_mappings': {'price': 'item_price', 'freight_value': 'shipping_fee', 'order_purchase_timestamp': 'purchased_at'}, 'files': ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']}


In [5]:

import csv
import json
import yaml
import math
import re
from pathlib import Path

project = Path(r"C:\tap-python-git")

with open(project / "config.yaml", encoding="utf-8") as file:
    config = yaml.safe_load(file)

data_folder = project / config["paths"]["data_dir"]
output_folder = project / config["paths"]["output_dir"]
output_folder.mkdir(exist_ok=True)

input_file = data_folder / "olist_order_items_dataset.csv"

clean_records = []
rejected_count = 0

with open(input_file, encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        try:
            price = float(row["price"])
            freight = float(row["freight_value"])

            if not math.isfinite(price) or not math.isfinite(freight):
                raise ValueError("Invalid numeric value")

            if price <= config["thresholds"]["min_price"] or freight < 0:
                raise ValueError("Invalid price or freight")

            for field in ["order_id", "product_id", "seller_id"]:
                if not re.fullmatch(r"[0-9a-fA-F]{32}", row[field]):
                    raise ValueError("Invalid ID")

            row["price"] = price
            row["freight_value"] = freight

            clean_row = {
                config["column_mappings"].get(key, key): value
                for key, value in row.items()
            }

            clean_records.append(clean_row)

        except (ValueError, TypeError, KeyError):
            rejected_count += 1

total = len(clean_records) + rejected_count

if total == 0 or not clean_records:
    raise ValueError("No valid records available")

if rejected_count / total > config["thresholds"]["max_rejection_rate"]:
    raise ValueError("Rejection threshold exceeded")

csv_path = output_folder / "clean_order_items.csv"
json_path = output_folder / "clean_order_items.json"

with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file, fieldnames=clean_records[0].keys()
    )
    writer.writeheader()
    writer.writerows(clean_records)

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(clean_records, file, indent=4)

print("Total records:", total)
print("Valid records:", len(clean_records))
print("Rejected records:", rejected_count)
print("CSV saved:", csv_path)
print("JSON saved:", json_path)


Total records: 576
Valid records: 576
Rejected records: 0
CSV saved: C:\tap-python-git\output\clean_order_items.csv
JSON saved: C:\tap-python-git\output\clean_order_items.json


In [7]:
import time

start_time = time.perf_counter()

duration = round(time.perf_counter() - start_time, 4)

In [8]:



import json
from datetime import datetime
from pathlib import Path

summary = {
    "pipeline": "Olist Data Pipeline",
    "source_file": input_file.name,
    "run_timestamp": datetime.now().isoformat(),
    "total_records": total,
    "valid_records": len(clean_records),
    "rejected_records": rejected_count,
    "rejection_percentage": round(
        rejected_count / total * 100, 2
    ),
    "duration_seconds": duration,
    "status": "SUCCESS"
}

summary_path = Path(r"C:\tap-python-git\summary.json")

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print(json.dumps(summary, indent=4))
print("Summary saved:", summary_path)



{
    "pipeline": "Olist Data Pipeline",
    "source_file": "olist_order_items_dataset.csv",
    "run_timestamp": "2026-11-24T14:23:36.580561",
    "total_records": 576,
    "valid_records": 576,
    "rejected_records": 0,
    "rejection_percentage": 0.0,
    "duration_seconds": 0.0001,
    "status": "SUCCESS"
}
Summary saved: C:\tap-python-git\summary.json


In [9]:

from pathlib import Path
from day14_manifest import (
    file_hash,
    load_manifest,
    already_processed
)

project = Path(r"C:\tap-python-git")
manifest_path = project / "manifest.json"

manifest = load_manifest(manifest_path)

input_file = (
    project / "data" / "olist_order_items_dataset.csv"
)

signature = (
    file_hash(input_file)
    + ":"
    + file_hash(project / "config.yaml")
)

if already_processed(
    manifest, input_file.name, signature
):
    print("SKIPPED:", input_file.name)
else:
    print("READY TO PROCESS:", input_file.name)

print("Previously processed files:", len(manifest))


READY TO PROCESS: olist_order_items_dataset.csv
Previously processed files: 0


In [11]:

import json
from day14_pipeline import run_pipeline

summary = run_pipeline(
    r"C:\tap-python-git\config.yaml"
)

print(json.dumps(summary, indent=4))


{
    "run_timestamp": "20261124_143845_700653",
    "processed_files": 9,
    "skipped_files": 0,
    "failed_files": [],
    "total_records": 5739,
    "valid_records": 5739,
    "rejected_records": 0,
    "duration_seconds": 0.167,
    "status": "SUCCESS",
    "log_file": "pipeline_20261124_143845_700653.log"
}


In [12]:
%pip install ruff

   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ------ --------------------------------- 1.6/10.4 MB 11.5 MB/s eta 0:00:01
   ----------------- ---------------------- 4.5/10.4 MB 13.5 MB/s eta 0:00:01
   ----------------------------- ---------- 7.6/10.4 MB 13.9 MB/s eta 0:00:01
   ---------------------------------- ----- 8.9/10.4 MB 12.3 MB/s eta 0:00:01
   ---------------------------------------- 10.4/10.4 MB 11.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from pathlib import Path

path = Path(r"C:\tap-python-git\day14_pipeline.py")

code = path.read_text(encoding="utf-8")

old = '''        if field.endswith("zip_code_prefix"):
            if not re.fullmatch(r"[0-9]{5}", value):
                raise ValueError("Invalid ZIP prefix")'''

new = '''        if field.endswith("zip_code_prefix") and not re.fullmatch(r"[0-9]{5}", value):
            raise ValueError("Invalid ZIP prefix")'''

if code.count(old) == 1:
    path.write_text(code.replace(old, new), encoding="utf-8")
    print("ZIP validation fixed successfully")
else:
    print("Matching code not found. Check the file.")

ZIP validation fixed successfully


In [22]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "ruff", "check",
        "day14_manifest.py",
        "day14_pipeline.py",
        "reader.py",
        "test_day12.py",
        "transformer.py",
        "validator.py"
    ],
    cwd=r"C:\tap-python-git",
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

All checks passed!


Exit code: 0
